# Memory-Constrained Distributed Trainer

**Audience:** ML practitioners who know Python and basic linear algebra.  
**Prerequisites:** NumPy, optimization basics, and the project README.  
**Learning goals:** implement the core algorithm, validate invariants, and evaluate a deterministic end-to-end run.

## Outline
1. Configure the reference system.
2. Execute its core training or simulation loop.
3. Verify numerical and behavioral assertions.
4. Try the exercise and inspect pitfalls.


In [1]:
from pathlib import Path
import sys, numpy as np
ROOT=Path.cwd()
if not (ROOT/'src').exists():
    ROOT=Path('/Users/shivamkumar/Documents/Codex/ml/implementations/mini-distributed-training-and-memory-constrained-trainer-from-sc')
sys.path.insert(0,str(ROOT/'src'))
np.set_printoptions(precision=3,suppress=True)


## 1. Inspect the memory budget
The model computes in FP16 inputs but accumulates gradients in FP32.


In [2]:
from memory_trainer.core import *
cfg=TrainerConfig(epochs=22)
x,y=data(cfg); p=init(cfg)
print(x.dtype, {k:v.shape for k,v in p.items()}, 'ZeRO shard lengths:', list(map(len,zero_shards(p,cfg.world_size))))


float16 {'w1': (6, 10), 'w2': (10, 1)} ZeRO shard lengths: [35, 35]


## 2. Accumulate, recompute, reduce, update
The backward pass recomputes hidden activations rather than retaining them.


In [3]:
out=train(cfg)
print('accuracy:',round(out['accuracy'],3),'first:',round(out['history'][0],3),'saved activations:',out['peak_saved_activations'])


accuracy: 0.896 first: 0.562 saved activations: 80


In [4]:
assert out['accuracy']>.82
assert out['history'][-1]>=out['history'][0]
print('memory-constrained training passed')


memory-constrained training passed


## Exercise
Change one capacity, rank, learning-rate, sample-size, or risk parameter in `config.json`. Predict the direction of the primary metric before rerunning.


In [5]:
# Answer scaffold: record the parameter, prediction, and observed metric.
exercise = {'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}
print(exercise)


{'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}


## Pitfalls and extensions
**Pitfall:** comparing runs with different seeds or compute budgets can masquerade as an algorithmic gain. Keep the seed and budget fixed.  
**Extension:** add a larger held-out scenario and report both quality and resource cost.
